## Table 1. Overall Performance.

In [2]:
import os
from tensorboard.backend.event_processing import event_accumulator
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pickle
import torch

def load_tensorboard_events(file_path):
    # Create an EventAccumulator to load the event file.
    # The 'size_guidance' argument can be used to control how many items to load for each type.
    ea = event_accumulator.EventAccumulator(
        file_path,
        size_guidance={
            'scalars': 1000,   # maximum number of scalar events to load per tag
            'images': 4,       # maximum number of image events to load per tag
            'histograms': 1,   # maximum number of histogram events to load per tag
            'compressedHistograms': 1,
            'audio': 1,
        }
    )
    ea.Reload()  # load events from file

    # Print available tags
    tags = ea.Tags()
    print("Available Tags:", tags)

    # For example, to print all scalar events for each scalar tag:
    if 'scalars' in tags:
        for tag in tags['scalars']:
            print(f"\nScalar values for tag: {tag}")
            events = ea.Scalars(tag)
            for event in events:
                print(f"Step: {event.step}, Value: {event.value}")
                

def load_tensorboard_metrics(file_path):
    ea = event_accumulator.EventAccumulator(
        file_path,
        size_guidance={
            'scalars': 100,   # maximum number of scalar events to load per tag
            'images': 4,       # maximum number of image events to load per tag
            'histograms': 1,   # maximum number of histogram events to load per tag
            'compressedHistograms': 1,
            'audio': 1,
        }
    )
    ea.Reload()  # load events from file
    tags = ea.Tags()
    metrics = tags['scalars']
    
    return_dict = {
        'best_model_epoch': None,
        'in-domain test metric (per sample)': None,
        'test metric (per sample)': None,
    }
    
    for tag in metrics:
        if 'in-domain test metric' in tag:
            return_dict['in-domain test metric (per sample)'] = ea.Scalars(tag)[0].value
            return_dict['best_model_epoch'] = ea.Scalars(tag)[0].step   
        elif 'test metric' in tag:
            return_dict['test metric (per sample)'] = ea.Scalars(tag)[0].value
    return return_dict  


def load_series_metrics(tensor_board_dir, model_metrics):
    unmatched_sub_fodlers = []
    sub_fodlers = os.listdir(tensor_board_dir)
    models = model_metrics.keys()
    for sub_folder in sub_fodlers:
        sub_folder_path = os.path.join(tensor_board_dir, sub_folder)
        files = os.listdir(sub_folder_path)
        # find the key that in the sub_folder
        key = None
        for model in models:
            if model in sub_folder:
                key = model
                break
        if key is None:
            unmatched_sub_fodlers.append(sub_folder)
            continue
        for file in files:
            if file.startswith("events.out.tfevents"):
                file_path = os.path.join(sub_folder_path, file)
                metrics = load_tensorboard_metrics(file_path)
                metrics['sub_folder'] = sub_folder
                model_metrics[key].append(metrics)
    return model_metrics, unmatched_sub_fodlers


def MPJPE_2D(predicts, labels, per_sample = True):
    # if the shape of the predicts and labels are : (batch_size, num_joints * 2)
    # reshape the predicts and labels to (batch_size, num_joints, dim)
    if isinstance(predicts, torch.Tensor):
        predicts = predicts.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()
    predicts = predicts.reshape(predicts.shape[0], -1, 2)
    labels = labels.reshape(labels.shape[0], -1, 2)
    per_sample_mpjpe = np.mean(np.abs(np.linalg.norm(predicts - labels, axis=2)), axis=1)
    if per_sample:
        return per_sample_mpjpe
    return np.mean(per_sample_mpjpe, axis=0)

def MPJPE_3D(predicts, labels, per_sample = True):
    # if the shape of the predicts and labels are : (batch_size, num_joints * 3) 
    # reshape the predicts and labels to (batch_size, num_joints, dim)
    if isinstance(predicts, torch.Tensor):
        predicts = predicts.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()
    predicts = predicts.reshape(predicts.shape[0], -1, 3)
    labels = labels.reshape(labels.shape[0], -1, 3)
    per_sample_mpjpe = np.mean(np.linalg.norm(predicts - labels, axis=2), axis=1)
    # if there is any value less than 0, print the index
    if np.any(per_sample_mpjpe < 0):
        print("less than 0")
        print(np.where(per_sample_mpjpe < 0))
    if per_sample:
        return per_sample_mpjpe
    return np.mean(per_sample_mpjpe, axis=0)

## WiFi Gait Recognition (GaitID dataset)

In [3]:
all_results_folder = 'logs/widarGait/'


results_files = [
    'widarGait_swin_t_swin_t_0218093555',
    'widarGait_swin_V2_t_swin_v2_t_0217100402',
    'widarGait_CRATEsmall_crate_small_0207210007',
    'widarGait_rfnet_rf_net_0211204707',
    'widarGait_stfnet_stfnet_standard_0212092119',
    'widarGait_slnet_slnet_standard_0212111504',
    'widarGait_widar3_widar3_standard_0218162010',
    'widarGait_rfcrate_small_rf_crate_small_0207103219',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE',
    'RF-Net', 
    'STFNet', 
    'SLNet',
    'Widar3.0', 
    'RF-CRATE'
    ]


accuracy_list = []
for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])

mean_acc = []
std_acc = []

for acc in accuracy_list:
    # acc is a list of accuracies for each fold
    acc = np.array(acc)
    mean_acc.append(np.mean(acc)*100)
    std_acc.append(np.std(acc)*100)

for index, name in enumerate(model_names):
    print(f'{name}: {mean_acc[index]:.2f} ± {std_acc[index]:.2f}')
    

Swin-T: 96.86 ± 3.11
Swin-T-V2: 92.70 ± 4.90
CRATE: 98.96 ± 1.83
RF-Net: 99.00 ± 1.88
STFNet: 95.30 ± 4.14
SLNet: 94.79 ± 4.14
Widar3.0: 82.01 ± 6.53
RF-CRATE: 98.98 ± 1.91


## WiFi Gesture Recognition (Widar3.0 Dataset)

In [4]:
# take average on both in-domain and cross-domain as the overall performance metric

indomain_acc = None
indomain_acc_std = None

crossdomain_acc = None
crossdomain_acc_std = None

HCI_overall_acc = None
HCI_overall_acc_std = None

all_results_folder = 'logs/widar3G6/'


results_files = [
    'widar3G6_swin_t_swin_t_0128162053_indomain_test',
    'widar3G6_swin_V2_t_swin_v2_t_0128190018_indomain_test',
    'widar3G6_CRATEsmall_crate_small_0207102454_indomain_test',
    'widar3G6_rfnet_rf_net_0128042953_indomain_test',
    'widar3G6_stfnet_stfnet_standard_0128144953_indomain_test',
    'widar3G6_slnet_slnet_standard_0128070238_indomain_test',
    'widar3G6_widar3_widar3_standard_0226182952_indomain_test',
    'widar3G6_rfcrate_small_rf_crate_small_0128000257_indomain_test',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE', 
    'RF-Net', 
    'STFNet', 
    'SLNet',
    'Widar3.0', 
    'RF-CRATE'
    ]

accuracy_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])

mean_acc = []
std_acc = []

for acc in accuracy_list:
    acc = np.array(acc)
    mean_acc.append(np.mean(acc))
    std_acc.append(np.std(acc))

indomain_acc = mean_acc
indomain_acc_std = std_acc


results_files = [
    'widar3G6_swin_t_swin_t_0128162053',
    'widar3G6_swin_V2_t_swin_v2_t_0128190018',
    'widar3G6_CRATEsmall_crate_small_0207102454',
    'widar3G6_rfnet_rf_net_0128042953',
    'widar3G6_stfnet_stfnet_standard_0128144953',
    'widar3G6_slnet_slnet_standard_0128070238',
    'widar3G6_widar3_widar3_standard_0226182952',
    'widar3G6_rfcrate_small_rf_crate_small_0128000257',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE',
    'RF-Net', 
    'STFNet', 
    'SLNet',
    'Widar3.0', 
    'RF-CRATE'
    ]


accuracy_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])


mean_acc = []
std_acc = []

for acc in accuracy_list:
    acc = np.array(acc)
    mean_acc.append(np.mean(acc))
    std_acc.append(np.std(acc))

crossdomain_acc = mean_acc
crossdomain_acc_std = std_acc

# take average on cross and in domain
HCI_overall_acc = (np.array(indomain_acc)*100 + np.array(crossdomain_acc)*100) / 2
HCI_overall_acc_std = (np.array(indomain_acc_std)*100 + np.array(crossdomain_acc_std)*100) / 2

print("overall results")
for index, name in enumerate(model_names):
    print(f'{name}: {HCI_overall_acc[index]:.2f} ± {HCI_overall_acc_std[index]:.2f}')


overall results
Swin-T: 84.25 ± 9.96
Swin-T-V2: 55.65 ± 18.45
CRATE: 75.20 ± 12.92
RF-Net: 55.33 ± 15.91
STFNet: 50.61 ± 17.48
SLNet: 73.67 ± 13.11
Widar3.0: 50.45 ± 18.29
RF-CRATE: 82.10 ± 10.41


In [5]:
all_results_folder = 'logs/widar3G16/'

remaining_overall_acc = None
remaining_overall_acc_std = None

results_files = [
    'widar3G16_swin_t_swin_t_0202205337',
    'widar3G16_swin_V2_t_swin_v2_t_0203135802',
    'widar3G16_CRATEsmall_crate_small_0213132224',
    'widar3G16_rfnet_rf_net_0130144515',
    'widar3G16_stfnet_stfnet_standard_0202130054',
    'widar3G16_slnet_slnet_standard_0131193320',
    'widar3G16_widar3_widar3_standard_0219020026',
    'widar3G16_rfcrate_small_rf_crate_small_0129144426',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE',
    'RF-Net', 
    'STFNet', 
    'SLNet',
    'Widar3.0', 
    'RF-CRATE'
    ]

accuracy_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])


mean_acc = []
std_acc = []

for acc in accuracy_list:
    # acc is a list of accuracies for each fold
    acc = np.array(acc)
    mean_acc.append(np.mean(acc))
    std_acc.append(np.std(acc))

for index, name in enumerate(model_names):
    print(f'{name}: {mean_acc[index]:.4f} ± {std_acc[index]:.4f}')

remaining_overall_acc = mean_acc
remaining_overall_acc_std = std_acc


Swin-T: 0.9104 ± 0.0514
Swin-T-V2: 0.8883 ± 0.0537
CRATE: 0.9055 ± 0.0501
RF-Net: 0.8542 ± 0.0620
STFNet: 0.5611 ± 0.0897
SLNet: 0.6802 ± 0.0804
Widar3.0: 0.6867 ± 0.0816
RF-CRATE: 0.9117 ± 0.0508


In [6]:
Overall_acc = (np.array(indomain_acc) + np.array(crossdomain_acc)) / 2
Overall_acc_std = (np.array(indomain_acc_std) + np.array(crossdomain_acc_std)) / 2

print("overall results")
for index, name in enumerate(model_names):
    print(f'{name}: {Overall_acc[index]:.4f} ± {Overall_acc_std[index]:.4f}')
    

overall results
Swin-T: 0.8425 ± 0.0996
Swin-T-V2: 0.5565 ± 0.1845
CRATE: 0.7520 ± 0.1292
RF-Net: 0.5533 ± 0.1591
STFNet: 0.5061 ± 0.1748
SLNet: 0.7367 ± 0.1311
Widar3.0: 0.5045 ± 0.1829
RF-CRATE: 0.8210 ± 0.1041


## WiFi 3D pose (WiP-pose Dataset, i.e., OctonetMini)

In [7]:
all_results_folder = 'logs/OctonetMini/'

results_files = [
    'OctonetMini_swin_t_swin_t_0207222621',
    'OctonetMini_swin_V2_t_swin_v2_t_0207230420', 
    'OctonetMini_CRATEsmall_crate_small_0207162345',
    'OctonetMini_rfcrate_small_ssr_2_rf_crate_small_0319170431',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2', 
    'CRATE',
    'RF-CRATE'
    ]


metric_list = []
predicts_list = []
labels_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        metric_list.append(results['metrics'])
        predicts_list.append(results['predicts'])
        labels_list.append(results['labels'])


mpjpe_list = []
for predicts, labels in zip(predicts_list, labels_list):
    mpjpe_temp = []
    for i in range(len(predicts)):
        mpjpe = MPJPE_3D(predicts[i], labels[i])  # one batch
        mpjpe_temp += mpjpe.tolist()
    mpjpe_list.append(mpjpe_temp)

mpjpe_mean = np.array([np.mean(mpjpe) for mpjpe in mpjpe_list])
std_mpjpe = np.array([np.std(mpjpe) for mpjpe in mpjpe_list])
for index, name in enumerate(model_names):
    print(f'{name}: {mpjpe_mean[index]/10:.2f} ± {std_mpjpe[index]/10:.2f}')


Swin-T: 28.70 ± 26.23
Swin-T-V2: 28.90 ± 25.99
CRATE: 28.66 ± 26.28
RF-CRATE: 28.12 ± 27.02


## WiFi Breath (WiP-breath Dataset, i.e., RPI)

In [8]:
all_results_folder = 'logs/RPI/'

results_files =[
    'RPI_swin_t_swin_t_0317220225_indomain_test',
    'RPI_swin_V2_t_swin_v2_t_0317220518_indomain_test',
    'RPI_rfnet_rf_net_0317215949_indomain_test',
    'RPI_CRATEsmall_crate_small_0317215143_indomain_test',
    'RPI_rfcrate_small_rf_crate_small_0317215801_indomain_test',
    
    ]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'RF-Net', 
    'CRATE',
    'RF-CRATE'
    ]


bpme_list = []  # beat per minute error

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        bpme_list.append(results['metrics'])


mean_bpme = []
std_bpme = []

for bpme in bpme_list:
    bpme = np.array(bpme)
    mean_bpme.append(np.mean(bpme))
    std_bpme.append(np.std(bpme))

for index, name in enumerate(model_names):
    print(f'{name}: {mean_bpme[index]:.2f} ± {std_bpme[index]:.2f}')



Swin-T: 2.44 ± 0.86
Swin-T-V2: 2.44 ± 0.77
RF-Net: 3.57 ± 0.98
CRATE: 2.44 ± 0.85
RF-CRATE: 2.44 ± 0.83


## mmWave 2D pose estimation (HuPR dataset)

In [10]:
all_results_folder = 'logs/HuPR/'

indomain_mpjpe = None
indomain_mpjpe_std = None

crossdomain_mpjpe = None
crossdomain_mpjpe_std = None


# this part is the cross-domain 
results_files = [
    'HuPR_swin_t_swin_t_0219044545',
    'HuPR_swin_V2_t_swin_v2_t_0219092304',
    'HuPR_CRATEsmall_crate_small_0220233208',
    'HuPR_rfnet_rf_net_0218200625',
    'HuPR_stfnet_stfnet_standard_0221095959',
    'HuPR_widar3_widar3_standard_0221123222',
    'HuPR_rfcrate_mini_rf_crate_mini_0326213330',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE',
    'RF-Net', 
    'STFNet', 
    'Widar3.0', 
    'RF-CRATE',
    ]

metric_list = []
predicts_list = []
labels_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        metric_list.append(results['metrics'])
        predicts_list.append(results['predicts'])
        labels_list.append(results['labels'])
        

mpjpe_list = []
for predicts, labels in zip(predicts_list, labels_list):
    mpjpe_temp = []
    for i in range(len(predicts)):
        mpjpe = MPJPE_2D(predicts[i], labels[i])  # one batch
        mpjpe_temp += mpjpe.tolist()
    mpjpe_list.append(mpjpe_temp)

mpjpe_mean = np.array([np.mean(mpjpe) for mpjpe in mpjpe_list])
mpjpe_std = np.array([np.std(mpjpe) for mpjpe in mpjpe_list])

crossdomain_mpjpe = mpjpe_mean
crossdomain_mpjpe_std = mpjpe_std


    
results_files = [
    'HuPR_swin_t_swin_t_0219044545_indomain_test',
    'HuPR_swin_V2_t_swin_v2_t_0219092304_indomain_test',
    'HuPR_CRATEsmall_crate_small_0220233208_indomain_test',
    'HuPR_rfnet_rf_net_0218200625_indomain_test',
    'HuPR_stfnet_stfnet_standard_0221095959_indomain_test',
    'HuPR_widar3_widar3_standard_0221123222_indomain_test',
    'HuPR_rfcrate_mini_rf_crate_mini_0326213330_indomain_test',
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'CRATE',
    'RF-Net', 
    'STFNet', 
    'Widar3.0', 
    'RF-CRATE',
    ]

metric_list = []
predicts_list = []
labels_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        metric_list.append(results['metrics'])
        predicts_list.append(results['predicts'])
        labels_list.append(results['labels'])
        

mpjpe_list = []
for predicts, labels in zip(predicts_list, labels_list):
    mpjpe_temp = []
    for i in range(len(predicts)):
        mpjpe = MPJPE_2D(predicts[i], labels[i])  # one batch
        mpjpe_temp += mpjpe.tolist()
    mpjpe_list.append(mpjpe_temp)

mpjpe_mean = np.array([np.mean(mpjpe) for mpjpe in mpjpe_list])
mpjpe_std = np.array([np.std(mpjpe) for mpjpe in mpjpe_list])

indomain_mpjpe = mpjpe_mean
indomain_mpjpe_std = mpjpe_std

overall_mpjpe = (indomain_mpjpe + crossdomain_mpjpe) / 2
overall_mpjpe_std = (indomain_mpjpe_std + crossdomain_mpjpe_std) / 2

print("overall results")
for index, name in enumerate(model_names):
    print(f'{name}: {overall_mpjpe[index]:.2f} ± {overall_mpjpe_std[index]:.2f}')


overall results
Swin-T: 10.61 ± 6.02
Swin-T-V2: 10.40 ± 6.54
CRATE: 24.93 ± 11.96
RF-Net: 8.37 ± 5.91
STFNet: 47.20 ± 21.44
Widar3.0: 62.40 ± 15.70
RF-CRATE: 17.09 ± 9.13


## UWB Activity Recognition (OperaNet)

In [11]:
all_results_folder = 'logs/OperaNet/UWB1/'

UWB1_overall_acc = None
UWB1_overall_acc_std = None

results_files = [
    'OPERAnet_UWB1_swin_t_swin_t_0127165251',
    'OPERAnet_UWB1_swin_V2_t_swin_v2_t_0127170307',
    'OPERAnet_UWB1_rfnet_rf_net_0127161932',
    'OPERAnet_UWB1_stfnet_stfnet_standard_0127162737',
    'OPERAnet_UWB1_CRATEsmall_crate_small_0127153655',
    'OPERAnet_UWB1_rfcrate_wide_tiny_rf_crate_wide_tiny_0327015951'
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'RF-Net', 
    'STFNet', 
    'CRATE',
    'RF-CRATE'
    ]


accuracy_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])


mean_acc = []
std_acc = []

for acc in accuracy_list:
    # acc is a list of accuracies for each fold
    acc = np.array(acc)
    mean_acc.append(np.mean(acc))
    std_acc.append(np.std(acc))


UWB1_overall_acc = np.array(mean_acc)*100
UWB1_overall_acc_std = np.array(std_acc)*100

print("overall results")
for index, name in enumerate(model_names):
    print(f'{name}: {UWB1_overall_acc[index]:.2f} ± {UWB1_overall_acc_std[index]:.2f}')
    

overall results
Swin-T: 66.07 ± 9.94
Swin-T-V2: 61.16 ± 8.97
RF-Net: 54.02 ± 7.96
STFNet: 56.25 ± 8.01
CRATE: 64.29 ± 8.97
RF-CRATE: 56.25 ± 10.30


In [12]:
all_results_folder = 'logs/OperaNet/UWB2/'

UWB2_overall_acc = None
UWB2_overall_acc_std = None


results_files = [
    'OPERAnet_UWB2_swin_t_swin_t_0127162959',
    'OPERAnet_UWB2_swin_V2_t_swin_v2_t_0127163328',
    'OPERAnet_UWB2_rfnet_rf_net_0127161012',
    'OPERAnet_UWB2_stfnet_stfnet_standard_0127161400',
    'OPERAnet_UWB2_CRATEsmall_crate_small_0127153639',
    'OPERAnet_UWB2_rfcrate_wide_tiny_rf_crate_wide_tiny_0327021616'
]

model_names = [
    'Swin-T', 
    'Swin-T-V2',
    'RF-Net', 
    'STFNet', 
    'CRATE',
    'RF-CRATE'
    ]


accuracy_list = []

for results_file in results_files:
    with open(all_results_folder + results_file + '.pkl', 'rb') as handle:
        results = pickle.load(handle)
        accuracy_list.append(results['metrics'])


mean_acc = []
std_acc = []

for acc in accuracy_list:
    # acc is a list of accuracies for each fold
    acc = np.array(acc)
    mean_acc.append(np.mean(acc))
    std_acc.append(np.std(acc))


UWB2_overall_acc = np.array(mean_acc)*100
UWB2_overall_acc_std = np.array(std_acc)*100

print("overall results")
for index, name in enumerate(model_names):
    print(f'{name}: {UWB2_overall_acc[index]:.4f} ± {UWB2_overall_acc_std[index]:.4f}')


overall results
Swin-T: 67.9688 ± 8.3780
Swin-T-V2: 64.0625 ± 4.6875
RF-Net: 61.7188 ± 2.5911
STFNet: 60.9375 ± 7.1603
CRATE: 51.5625 ± 8.9759
RF-CRATE: 63.8889 ± 10.5318


In [13]:
Overall_acc = None
Overall_acc_std = None

Overall_acc = (np.array(UWB1_overall_acc) + np.array(UWB2_overall_acc)) / 2
Overall_acc_std = (np.array(UWB1_overall_acc_std) + np.array(UWB2_overall_acc_std)) / 2

for index, name in enumerate(model_names):
    print(f'{name}: {Overall_acc[index]:.2f} ± {Overall_acc_std[index]:.2f}')


Swin-T: 67.02 ± 9.16
Swin-T-V2: 62.61 ± 6.83
RF-Net: 57.87 ± 5.28
STFNet: 58.59 ± 7.59
CRATE: 57.92 ± 8.97
RF-CRATE: 60.07 ± 10.41
